# dLLM vs LLM Summarisation Speed Profiling

**Goal:** Find dLLM generation configs that beat autoregressive LLM sequential summarisation speed.

**Hardware:** 2×T4 (16 GB each)  
**Models:** Fast-dLLM v2 1.5B vs Qwen2.5-1.5B-Instruct  
**Strata:** 2, 3, 4, 5, 6, 7, 8, 9 files per commit × 5 tasks + 10 tasks with 10-21 files = 50 tasks

In [ ]:
!pip install -q transformers==4.57 torch accelerate sentencepiece protobuf

In [ ]:
import torch
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_mem / 1024**3:.1f} GB")

## 1. Write library files

In [ ]:
!mkdir -p lib

In [ ]:
%%writefile lib/generation_functions.py
"""
Fast-dLLM v2 generation functions.

Sourced from: https://github.com/NVlabs/Fast-dLLM/blob/main/v2/generation_functions.py
Only batch_sample is included; used for batched commit message generation.

Copyright 2025 NVIDIA CORPORATION & AFFILIATES
Licensed under the Apache License, Version 2.0
SPDX-License-Identifier: Apache-2.0
"""

import torch


def _slice_kv_cache(cache, keep_mask):
    """Slice the batch dimension of a KV cache, compatible with multiple transformers versions."""
    if cache is None:
        return None
    if hasattr(cache, 'batch_select_indices'):
        indices = keep_mask.nonzero(as_tuple=False).squeeze(-1)
        cache.batch_select_indices(indices)
        return cache
    if hasattr(cache, 'key_cache'):
        for i in range(len(cache.key_cache)):
            cache.key_cache[i] = cache.key_cache[i][keep_mask]
            cache.value_cache[i] = cache.value_cache[i][keep_mask]
        return cache
    if isinstance(cache, (list, tuple)):
        return tuple((k[keep_mask], v[keep_mask]) for k, v in cache)
    return cache


FAST_DLLM_MASK_ID = 151665
FAST_DLLM_STOP_TOKEN = 151645


class Fast_dLLM_QwenForCausalLM:
    """Mixin class — methods are bound to the model via types.MethodType."""

    @torch.no_grad()
    def batch_sample(
        self,
        input_ids,
        tokenizer,
        block_size,
        max_new_tokens,
        small_block_size,
        min_len,
        seq_len,
        mask_id=FAST_DLLM_MASK_ID,
        threshold=0.95,
        stop_token=FAST_DLLM_STOP_TOKEN,
        use_block_cache=False,
        top_p=0.95,
        temperature=0.0,
    ):
        num_blocks = max_new_tokens // block_size + seq_len.max().item() // block_size
        batch_size = input_ids.shape[0]

        if min_len > block_size:
            output = self.forward(
                input_ids=input_ids[:, :(min_len // block_size * block_size)],
                use_cache=True,
                update_past_key_values=True,
                block_size=block_size,
            )
            logits, past_key_values = output.logits, output.past_key_values
            if min_len % block_size == 0:
                predict_sample_idx = (seq_len == min_len)
                predict_logits = logits[predict_sample_idx, -1:, :]
                next_token = predict_logits.argmax(dim=-1)
                if input_ids.shape[1] <= min_len:
                    input_ids = torch.cat([input_ids, next_token], dim=1)
                else:
                    input_ids[predict_sample_idx, min_len] = next_token.squeeze(dim=-1)
        else:
            past_key_values = None

        seq_block_idx = seq_len // block_size
        finished_flag = torch.zeros((batch_size,), device=self.device, dtype=torch.bool)

        start_block_idx = min_len // block_size
        num_small_blocks = block_size // small_block_size

        sample_indices = torch.arange(batch_size, device=self.device)
        finished_samples = {}
        total_steps = 0

        for block_idx in range(start_block_idx, num_blocks):
            if finished_flag.all():
                break

            if (seq_block_idx == block_idx).all():
                x_init = mask_id * torch.ones(
                    (input_ids.shape[0], block_size - input_ids.shape[1] % block_size),
                    device=self.device,
                    dtype=torch.long,
                )
                x_init = torch.cat([input_ids, x_init], dim=1)
                input_ids = x_init
            else:
                x_init = input_ids[:, :(block_idx + 1) * block_size]

            x_init[finished_flag, -block_size:] = tokenizer.pad_token_id
            x_t = x_init.clone()
            step = 0
            block_past_key_values = None

            while True:
                mask_idx = (x_t[:, -block_size:] == mask_id)
                if mask_idx.sum() == 0:
                    for sample_idx in range(x_t.shape[0]):
                        if finished_flag[sample_idx] and seq_len[sample_idx] < (block_idx + 1) * block_size:
                            stop_token_idx = (x_t[sample_idx, seq_len[sample_idx]:] == stop_token).nonzero()[0][0]
                            x_t[sample_idx, seq_len[sample_idx] + stop_token_idx + 1:] = tokenizer.pad_token_id
                    if finished_flag.all():
                        break
                    output = self.forward(
                        input_ids=x_t[:, -block_size:],
                        use_cache=True,
                        past_key_values=past_key_values,
                        update_past_key_values=True,
                        block_size=block_size,
                    )
                    logits, past_key_values = output.logits, output.past_key_values
                    next_token = logits[:, -1:, :].argmax(dim=-1)
                    next_token[finished_flag] = tokenizer.pad_token_id
                    x_t = torch.cat([x_t, next_token], dim=1)
                    step += 1
                    break

                for small_block_idx in range(num_small_blocks):
                    small_block_start_idx = small_block_idx * small_block_size
                    small_block_end_idx = small_block_start_idx + small_block_size

                    start = -block_size + small_block_start_idx
                    end = None if block_size == small_block_end_idx else -block_size + small_block_end_idx

                    while True:
                        mask_idx = (x_t[:, -block_size:] == mask_id)
                        if mask_idx[:, start:end].sum() == 0:
                            break

                        if use_block_cache:
                            if block_past_key_values is None or (x_t[:, -block_size + small_block_start_idx] == mask_id).any():
                                output = self.forward(
                                    input_ids=x_t[:, -block_size:],
                                    use_cache=True,
                                    past_key_values=past_key_values,
                                    update_past_key_values=False,
                                    use_block_cache=True,
                                    block_size=block_size,
                                )
                                logits, block_past_key_values = output.logits, output.block_past_key_values
                                logits = torch.cat([logits[:, :1, :], logits[:, :-1, :]], dim=1)
                                logits = logits[:, start:end]
                            else:
                                logits = self.forward(
                                    input_ids=x_t[:, start:end],
                                    use_cache=True,
                                    past_key_values=past_key_values,
                                    update_past_key_values=False,
                                    use_block_cache=True,
                                    block_past_key_values=block_past_key_values,
                                    replace_position=small_block_start_idx,
                                ).logits
                                logits = torch.cat([logits[:, :1, :], logits[:, :-1, :]], dim=1)
                        else:
                            logits = self.forward(
                                input_ids=x_t[:, -block_size:],
                                use_cache=True,
                                past_key_values=past_key_values,
                                update_past_key_values=False,
                                block_size=block_size,
                            ).logits
                            logits = torch.cat([logits[:, :1, :], logits[:, :-1, :]], dim=1)
                            logits = logits[:, start:end]

                        x_1, p_1t = self.sample_with_top_p(logits, top_p=top_p, temperature=temperature)
                        x1_p = torch.squeeze(torch.gather(p_1t, dim=-1, index=torch.unsqueeze(x_1, -1)), -1)
                        x1_p = torch.where(mask_idx[:, start:end], x1_p, -torch.inf)

                        unmask_idx = (x1_p > threshold)
                        max_prob_idx = x1_p.argmax(dim=-1)
                        unmask_idx[torch.arange(x_1.shape[0]), max_prob_idx] = True
                        unmask_idx = unmask_idx & mask_idx[:, start:end]

                        x_t[:, start:end][unmask_idx] = x_1[unmask_idx]

                        finished_row_flags = ((x_1 == stop_token) & unmask_idx).any(dim=1)
                        finished_flag = finished_flag | finished_row_flags

                        step += 1

            total_steps += step

            if input_ids.shape[1] == x_t.shape[1]:
                input_ids = x_t
            else:
                input_ids[:, :(block_idx + 1) * block_size] = x_t[:, :-1]
                if (seq_block_idx == block_idx).all():
                    input_ids = torch.cat([input_ids, x_t[:, -1:]], dim=1)
                else:
                    if input_ids.shape[1] <= (block_idx + 1) * block_size:
                        input_ids = x_t
                    else:
                        input_ids[seq_block_idx == block_idx, (block_idx + 1) * block_size] = \
                            x_t[seq_block_idx == block_idx, (block_idx + 1) * block_size]

            seq_block_idx[seq_block_idx == block_idx] = block_idx + 1

            if finished_flag.any():
                for sample_idx in range(x_t.shape[0]):
                    if finished_flag[sample_idx]:
                        original_idx = sample_indices[sample_idx].item()
                        finished_samples[original_idx] = x_t[sample_idx:sample_idx + 1].clone().squeeze(dim=0)

                sample_indices = sample_indices[~finished_flag]
                input_ids = input_ids[~finished_flag]
                seq_block_idx = seq_block_idx[~finished_flag]
                seq_len = seq_len[~finished_flag]
                x_t = x_t[~finished_flag]

                past_key_values = _slice_kv_cache(past_key_values, ~finished_flag)

                finished_flag = finished_flag[~finished_flag]

        # Add unfinished samples (max_new_tokens reached)
        if len(finished_samples) < batch_size:
            for sample_idx in range(x_t.shape[0]):
                original_idx = sample_indices[sample_idx].item()
                finished_samples[original_idx] = x_t[sample_idx:sample_idx + 1].clone().squeeze(dim=0)

        assert len(finished_samples) == batch_size
        return finished_samples, total_steps

In [ ]:
%%writefile lib/diff_utils.py
"""
diff_utils.py — Utilities for splitting unified git diffs into per-file chunks.
"""

import re
from typing import Optional

_DIFF_START = "--- START OF CODE DIFF ---"
_DIFF_END = "--- END OF CODE DIFF ---"
_HEADER_RE = re.compile(r"^diff --git a/.+ b/(.+)$", re.MULTILINE)


def extract_diff_content(user_content: str) -> Optional[str]:
    start = user_content.find(_DIFF_START)
    end = user_content.find(_DIFF_END)
    if start == -1 or end == -1 or end <= start:
        return None
    return user_content[start + len(_DIFF_START) : end].strip()


def split_diff_by_file(diff: str) -> list[tuple[str, str]]:
    matches = list(_HEADER_RE.finditer(diff))
    if not matches:
        return []
    result: list[tuple[str, str]] = []
    for i, match in enumerate(matches):
        filename = match.group(1).strip()
        start_pos = match.start()
        end_pos = matches[i + 1].start() if i + 1 < len(matches) else len(diff)
        file_diff = diff[start_pos:end_pos].strip()
        result.append((filename, file_diff))
    return result


def get_per_file_diffs(task: dict) -> list[tuple[str, str]]:
    user_content = ""
    for msg in task.get("messages", []):
        if msg.get("role") == "user":
            user_content = msg.get("content", "")
            break
    if not user_content:
        return []
    diff = extract_diff_content(user_content)
    if diff is None:
        return [("(full diff)", user_content)]
    per_file = split_diff_by_file(diff)
    if not per_file:
        return [("(full diff)", diff)]
    return per_file


_SUMMARY_SYSTEM = (
    "You are a developer. Given the diff for a single file in a commit, "
    "write a concise technical summary (2-4 sentences) of what changed and why. "
    "Focus on the semantic meaning of the changes, not line-by-line description."
)

_CMG_SYSTEM = (
    "You are a developer, and your task is to write a concise commit message "
    "based on per-file summaries of the code changes in a commit.\n"
    "## Output Format:\n"
    "A concise commit message describing the code changes as plain text, "
    "wrapped in <msg> </msg> tags. Nothing else after the </msg> tag.\n"
    "Example output: <msg>Fix indefinite loading for users</msg>\n"
    "Example output: <msg>feat(server): Add new API endpoint for user registration</msg>"
)


def build_summary_messages(filename: str, file_diff: str, max_diff_chars: int | None = None) -> list[dict]:
    if max_diff_chars is not None and len(file_diff) > max_diff_chars:
        cut = file_diff.rfind("\n", 0, max_diff_chars)
        if cut == -1:
            cut = max_diff_chars
        file_diff = file_diff[:cut] + "\n... [diff truncated]"
    return [
        {"role": "system", "content": _SUMMARY_SYSTEM},
        {
            "role": "user",
            "content": (
                f"File: {filename}\n\n"
                "--- START OF FILE DIFF ---\n"
                f"{file_diff}\n"
                "--- END OF FILE DIFF ---\n\n"
                "Write a concise technical summary of the changes in this file."
            ),
        },
    ]


def build_cmg_messages(file_summaries: list[tuple[str, str]], original_system: Optional[str] = None) -> list[dict]:
    summary_block = "\n\n".join(f"### {fn}\n{s}" for fn, s in file_summaries)
    user_content = (
        "The following are concise summaries of each file changed in a commit. "
        "Based on these summaries, write ONLY the appropriate one sentence "
        "commit message, between <msg> </msg> tags.\n\n"
        "## File Summaries:\n\n"
        f"{summary_block}"
    )
    system = original_system if original_system else _CMG_SYSTEM
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user_content},
    ]

## 2. Write benchmark script

In [ ]:
%%writefile bench_dllm_batch.py
# >>> PASTE the contents of: profiling/bench_dllm_batch.py <<<

## 3. Upload pre-built tasks

Upload the pre-built `tasks_profiling_50.jsonl` file as a Kaggle dataset input.  
Build it locally with:
```bash
python build_tasks/build_tasks.py --dataset datasets/ApacheCM/test.jsonl --tasks build_tasks/tasks_profiling.jsonl --min-diff-length 2500 --max-diff-length 8500 --stats
python build_tasks/select_best_tasks.py
```

In [ ]:
import shutil, os

# Copy the pre-built tasks file from Kaggle input
TASKS_INPUT = "/kaggle/input/dllm-profiling/tasks_profiling_50.jsonl"
TASKS_OUTPUT = "tasks_profiling_50.jsonl"

shutil.copy(TASKS_INPUT, TASKS_OUTPUT)
print(f"Tasks ready: {TASKS_OUTPUT}")

# Quick line count
with open(TASKS_OUTPUT) as f:
    n = sum(1 for _ in f)
print(f"Total tasks: {n}")

In [ ]:
# Quick stats on file-count distribution
import json, sys
sys.path.insert(0, "lib")
from diff_utils import get_per_file_diffs

tasks = [json.loads(l) for l in open(TASKS_OUTPUT) if l.strip()]
file_counts = [len(get_per_file_diffs(t)) for t in tasks]

from collections import Counter
fc = Counter(file_counts)
targets = [4, 8, 16, 24, 32, 36]
print(f"Total tasks: {len(tasks)}")
print(f"\nFile-count availability for targets:")
for t in targets:
    lo = t if t <= 5 else int(t * 0.7)
    hi = t if t <= 5 else int(t * 1.3)
    available = sum(v for k, v in fc.items() if lo <= k <= hi)
    print(f"  target={t:>3}  range=[{lo},{hi}]  available={available}")

## 4. Run the profiling benchmark

Grid search: `block_size` × `threshold` × `max_new_tokens` combinations.  
File count strata: 2, 3, 4, 5, 6, 7, 8, 9, 10+ — 5 tasks each.  
Both models on `cuda:0` (shared GPU, fair comparison).

In [ ]:
OUTPUT_JSON = "profiling_results.json"

!python bench_dllm_batch.py \
    -i {TASKS_OUTPUT} \
    --file-counts 2,3,4,5,6,7,8,9,10,15,21 \
    --tasks-per-stratum 5 \
    --device cuda:0 \
    --llm-device cuda:1 \
    --llm-model Qwen/Qwen2.5-1.5B-Instruct \
    --dllm-model Efficient-Large-Model/Fast_dLLM_v2_1.5B \
    --block-sizes 32,64 \
    --small-block-sizes 8,16 \
    --thresholds 0.8,0.6,0.4 \
    --max-new-tokens 128,256 \
    --batch-sizes 1,4,8,16,32,36 \
    --max-diff-chars 600 \
    --no-summaries \
    --use-all-files \
    --output-file {OUTPUT_JSON}

## 5. Load results & build analysis dataframe

In [ ]:
import json
import pandas as pd
import numpy as np

with open(OUTPUT_JSON) as f:
    raw_results = json.load(f)

# Flatten into per-(task, config, batch_size) rows
rows = []
for r in raw_results:
    base = {
        "task_id": r["task_id"],
        "n_files": r["n_files"],
        "block_size": r["block_size"],
        "small_block_size": r["small_block_size"],
        "threshold": r["threshold"],
        "max_new_tokens": r["max_new_tokens"],
        "config": r["config"],
        "dllm_seq_total_s": r["seq_total"],
        "llm_seq_total_s": r.get("llm_total"),
        "dllm_seq_vs_llm": (r.get("llm_total") or 0) / r["seq_total"] if r["seq_total"] > 0 else None,
    }
    for bs_str, bdata in r["batches"].items():
        row = {**base}
        row["batch_size"] = int(bs_str)
        row["dllm_batch_wall_s"] = bdata.get("wall_s")
        row["speedup_vs_dllm_seq"] = bdata.get("speedup_vs_dllm_seq")
        row["speedup_vs_llm_seq"] = bdata.get("speedup_vs_llm_seq")
        row["oom"] = bdata.get("oom", False)
        row["total_steps"] = bdata.get("steps")
        gen_tok = bdata.get("gen_tokens", [])
        row["total_gen_tokens"] = sum(gen_tok) if gen_tok else 0
        if row["dllm_batch_wall_s"] and row["dllm_batch_wall_s"] > 0 and row["total_gen_tokens"]:
            row["dllm_batch_tps"] = row["total_gen_tokens"] / row["dllm_batch_wall_s"]
        else:
            row["dllm_batch_tps"] = None
        if row["total_steps"] and row["total_steps"] > 0 and row["total_gen_tokens"]:
            row["tokens_per_step"] = row["total_gen_tokens"] / row["total_steps"]
        else:
            row["tokens_per_step"] = None
        rows.append(row)

df = pd.DataFrame(rows)
df = df[~df["oom"]].copy()  # drop OOM rows
print(f"DataFrame: {len(df)} rows, {df['task_id'].nunique()} tasks")
df.head()

In [ ]:
# Save full dataframe as machine-readable CSV + JSON
df.to_csv("profiling_dataframe.csv", index=False)
df.to_json("profiling_dataframe.json", orient="records", indent=2)
print("Saved: profiling_dataframe.csv, profiling_dataframe.json")

## 6. Key metrics summary

In [ ]:
# Best speedup vs LLM per file-count stratum
best = (
    df[df["speedup_vs_llm_seq"].notna()]
    .groupby("n_files")
    .apply(lambda g: g.loc[g["speedup_vs_llm_seq"].idxmax()])
    .reset_index(drop=True)
)

print("\n" + "="*80)
print("BEST dLLM-batch SPEEDUP vs LLM-seq BY FILE COUNT")
print("="*80)
cols = ["n_files", "batch_size", "block_size", "small_block_size", "threshold",
        "max_new_tokens", "speedup_vs_llm_seq", "speedup_vs_dllm_seq",
        "dllm_batch_wall_s", "llm_seq_total_s", "tokens_per_step"]
print(best[cols].to_string(index=False))

# Crossover analysis
crossover = best[best["speedup_vs_llm_seq"] >= 1.0]
if len(crossover) > 0:
    print(f"\n✓ dLLM beats LLM starting at {crossover['n_files'].min()} files")
else:
    print("\n✗ dLLM never beats LLM in any stratum (need different configs)")

In [ ]:
# Aggregate stats per (n_files, block_size, threshold)
agg = (
    df[df["speedup_vs_llm_seq"].notna()]
    .groupby(["n_files", "block_size", "small_block_size", "threshold", "max_new_tokens", "batch_size"])
    .agg(
        mean_speedup_vs_llm=("speedup_vs_llm_seq", "mean"),
        max_speedup_vs_llm=("speedup_vs_llm_seq", "max"),
        mean_tokens_per_step=("tokens_per_step", "mean"),
        mean_dllm_tps=("dllm_batch_tps", "mean"),
        count=("speedup_vs_llm_seq", "count"),
    )
    .reset_index()
    .sort_values("mean_speedup_vs_llm", ascending=False)
)
print("\nTop 20 configs by mean speedup vs LLM:")
print(agg.head(20).to_string(index=False))

## 7. Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

In [ ]:
# Plot 1: Speedup vs LLM by file count (best config per stratum)
fig, ax = plt.subplots(figsize=(10, 5))

best_per_nf = (
    df[df["speedup_vs_llm_seq"].notna()]
    .groupby("n_files")["speedup_vs_llm_seq"]
    .agg(["max", "mean", "median"])
    .reset_index()
)

ax.bar(best_per_nf["n_files"] - 0.8, best_per_nf["max"], width=1.6,
       alpha=0.7, color="steelblue", label="Best config")
ax.bar(best_per_nf["n_files"] + 0.8, best_per_nf["median"], width=1.6,
       alpha=0.5, color="orange", label="Median config")
ax.axhline(1.0, color="red", linestyle="--", linewidth=1.5, label="Break-even (1.0×)")
ax.set_xlabel("Number of files in commit")
ax.set_ylabel("Speedup (dLLM-batch / LLM-seq)")
ax.set_title("dLLM Batched vs LLM Sequential — Speedup by Commit Complexity")
ax.legend()
ax.set_xticks(best_per_nf["n_files"])
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("plot_speedup_by_filecount.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 2: Heatmap — threshold vs batch_size (averaged over tasks & other params)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, bs_val in enumerate([32, 64]):
    subset = df[(df["block_size"] == bs_val) & df["speedup_vs_llm_seq"].notna()]
    if subset.empty:
        continue
    pivot = subset.pivot_table(
        values="speedup_vs_llm_seq",
        index="threshold",
        columns="batch_size",
        aggfunc="mean",
    )
    ax = axes[idx]
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0.3, vmax=2.5)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.1f}" for v in pivot.index])
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Threshold")
    ax.set_title(f"Mean Speedup vs LLM — block_size={bs_val}")
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im, ax=axes, shrink=0.8, label="Speedup (×)")
plt.suptitle("Speedup Heatmap: Threshold × Batch Size", fontsize=12)
plt.tight_layout()
plt.savefig("plot_heatmap_threshold_batchsize.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 3: Tokens per step vs threshold (shows parallelism efficiency)
fig, ax = plt.subplots(figsize=(8, 5))

for bs_val in sorted(df["block_size"].unique()):
    subset = df[(df["block_size"] == bs_val) & df["tokens_per_step"].notna()]
    if subset.empty:
        continue
    grouped = subset.groupby("threshold")["tokens_per_step"].mean()
    ax.plot(grouped.index, grouped.values, "o-", label=f"block_size={bs_val}")

ax.set_xlabel("Threshold")
ax.set_ylabel("Mean tokens / diffusion step")
ax.set_title("dLLM Parallelism Efficiency vs Unmasking Threshold")
ax.legend()
ax.grid(alpha=0.3)
ax.invert_xaxis()
plt.tight_layout()
plt.savefig("plot_tokens_per_step.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 4: Wall time comparison — dLLM batch vs LLM seq vs dLLM seq
fig, ax = plt.subplots(figsize=(10, 5))

best_rows = (
    df[df["speedup_vs_llm_seq"].notna()]
    .sort_values("speedup_vs_llm_seq", ascending=False)
    .drop_duplicates(subset=["n_files"], keep="first")
    .sort_values("n_files")
)

x = np.arange(len(best_rows))
width = 0.25

ax.bar(x - width, best_rows["llm_seq_total_s"], width, label="LLM sequential", color="tab:red", alpha=0.8)
ax.bar(x, best_rows["dllm_seq_total_s"], width, label="dLLM sequential", color="tab:orange", alpha=0.8)
ax.bar(x + width, best_rows["dllm_batch_wall_s"], width, label="dLLM batched (best)", color="tab:green", alpha=0.8)

ax.set_xlabel("Number of files")
ax.set_ylabel("Wall time (seconds)")
ax.set_title("Summarisation Wall Time: LLM-seq vs dLLM-seq vs dLLM-batch")
ax.set_xticks(x)
ax.set_xticklabels(best_rows["n_files"].values)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("plot_wall_time_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 5: Speedup scaling curve with confidence band
fig, ax = plt.subplots(figsize=(10, 5))

best_batch = (
    df[(df["speedup_vs_llm_seq"].notna()) & (df["batch_size"] == df["n_files"])]
)
if best_batch.empty:
    best_batch = df[df["speedup_vs_llm_seq"].notna()].copy()
    best_batch = best_batch[best_batch["batch_size"] <= best_batch["n_files"]]
    best_batch = best_batch.sort_values("batch_size", ascending=False).drop_duplicates(
        subset=["task_id", "block_size", "threshold", "max_new_tokens"], keep="first"
    )

grouped = best_batch.groupby("n_files")["speedup_vs_llm_seq"].agg(["mean", "std", "max", "min"])
grouped = grouped.reset_index()

ax.fill_between(grouped["n_files"], grouped["min"], grouped["max"], alpha=0.15, color="steelblue")
ax.plot(grouped["n_files"], grouped["mean"], "o-", color="steelblue", linewidth=2, label="Mean speedup")
ax.plot(grouped["n_files"], grouped["max"], "^--", color="green", alpha=0.7, label="Best config")
ax.axhline(1.0, color="red", linestyle="--", linewidth=1.5, label="Break-even")

ax.set_xlabel("Number of files in commit")
ax.set_ylabel("Speedup (dLLM-batch / LLM-seq)")
ax.set_title("Speedup Scaling Curve — dLLM batch vs LLM sequential")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plot_speedup_scaling_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Plot 6: Parameter sensitivity — individual parameter effect
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

params = [
    ("threshold", "Threshold"),
    ("block_size", "Block Size"),
    ("small_block_size", "Small Block Size"),
    ("max_new_tokens", "Max New Tokens"),
]

valid = df[df["speedup_vs_llm_seq"].notna()]

for ax, (param, label) in zip(axes.flat, params):
    grouped = valid.groupby(param)["speedup_vs_llm_seq"].agg(["mean", "std"]).reset_index()
    ax.bar(range(len(grouped)), grouped["mean"], yerr=grouped["std"],
           capsize=4, color="steelblue", alpha=0.7)
    ax.set_xticks(range(len(grouped)))
    ax.set_xticklabels(grouped[param])
    ax.axhline(1.0, color="red", linestyle="--", alpha=0.7)
    ax.set_xlabel(label)
    ax.set_ylabel("Speedup vs LLM")
    ax.set_title(f"Effect of {label}")
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Parameter Sensitivity Analysis", fontsize=13)
plt.tight_layout()
plt.savefig("plot_parameter_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Export research report data

In [ ]:
# Build a comprehensive report dict
report = {
    "metadata": {
        "dllm_model": "Efficient-Large-Model/Fast_dLLM_v2_1.5B",
        "llm_model": "Qwen/Qwen2.5-1.5B-Instruct",
        "device": "NVIDIA T4 16GB",
        "n_gpus": 2,
        "file_count_strata": [2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 21],
        "tasks_per_stratum": 5,
        "total_tasks": int(df["task_id"].nunique()),
        "total_configs_tested": len(df),
        "diff_length_range": "2500-8500 chars",
    },
    "grid_params": {
        "block_sizes": sorted(df["block_size"].unique().tolist()),
        "small_block_sizes": sorted(df["small_block_size"].unique().tolist()),
        "thresholds": sorted(df["threshold"].unique().tolist()),
        "max_new_tokens": sorted(df["max_new_tokens"].unique().tolist()),
        "batch_sizes": sorted(df["batch_size"].unique().tolist()),
    },
    "best_per_stratum": best[cols].to_dict(orient="records") if len(best) else [],
    "overall_stats": {
        "max_speedup_vs_llm": float(df["speedup_vs_llm_seq"].max()) if df["speedup_vs_llm_seq"].notna().any() else None,
        "mean_speedup_vs_llm": float(df["speedup_vs_llm_seq"].mean()) if df["speedup_vs_llm_seq"].notna().any() else None,
        "pct_configs_beating_llm": float((df["speedup_vs_llm_seq"] > 1.0).mean() * 100) if df["speedup_vs_llm_seq"].notna().any() else 0,
        "mean_tokens_per_step": float(df["tokens_per_step"].mean()) if df["tokens_per_step"].notna().any() else None,
        "crossover_file_count": int(crossover["n_files"].min()) if len(crossover) > 0 else None,
    },
    "top10_configs": agg.head(10).to_dict(orient="records"),
}

with open("profiling_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print("Saved: profiling_report.json")
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"  Max speedup vs LLM:        {report['overall_stats']['max_speedup_vs_llm']:.2f}×")
print(f"  Mean speedup vs LLM:       {report['overall_stats']['mean_speedup_vs_llm']:.2f}×")
print(f"  % configs beating LLM:     {report['overall_stats']['pct_configs_beating_llm']:.1f}%")
print(f"  Mean tokens/step:          {report['overall_stats']['mean_tokens_per_step']:.2f}")

In [ ]:
# List all output artifacts
import glob
artifacts = glob.glob("profiling_*") + glob.glob("plot_*")
print("Output artifacts:")
for f in sorted(artifacts):
    size = os.path.getsize(f) / 1024
    print(f"  {f:<45} {size:.1f} KB")